# IsoGrowingNCA Chameleon Equivariance

This notebook evaluates whether the LPPN decoder breaks equivariance for an isotropic growing NCA trained on `data/morphology_png/chameleon.png`.

Train the two configs first, or point `EXPERIMENTS` at existing checkpoint directories:

```bash
python train.py --config configs/nca2d/growing_isonca_chameleon_coords.yaml --test
python train.py --config configs/nca2d/growing_isonca_chameleon_no_coords.yaml --test
```

The notebook compares `rollout(transform(seed))` against `transform(rollout(seed))` after rendering. Exact 90-degree rotations and flips are interpolation-free; arbitrary angles use bilinear resampling and should be interpreted separately.


In [ ]:
from pathlib import Path
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from IPython.display import display

from logger.disabled import DisabledLogger
from training.common import load_checkpoint_pair, load_yaml
from training.tasks import get_task
from training.tasks.targets import append_target_name_to_experiment


In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
STEPS = 512
ANGLES = [30.0, 45.0, 60.0, 120.0]

EXPERIMENTS = {
    'coords': {
        'config': 'configs/nca2d/growing_isonca_chameleon_coords.yaml',
        'experiment_path': None,
        'suffix': '',
    },
    'no_coords': {
        'config': 'configs/nca2d/growing_isonca_chameleon_no_coords.yaml',
        'experiment_path': None,
        'suffix': '',
    },
}

TARGET_PATH = Path('data/morphology_png/chameleon.png')
if TARGET_PATH.exists():
    img = Image.open(TARGET_PATH).convert('RGBA')
    print(TARGET_PATH, img.size)
    display(img.resize((320, round(320 * img.height / img.width))))
else:
    print('Missing target:', TARGET_PATH)

print('device:', DEVICE)


In [ ]:
def default_experiment_path(config):
    config = copy.deepcopy(config)
    append_target_name_to_experiment(config)
    if 'experiment_path' in config:
        return Path(config['experiment_path'])
    return Path('experiments') / config['experiment_name'].replace(' ', '_')


def load_variant(entry):
    config = copy.deepcopy(load_yaml(entry['config']))
    config['device'] = DEVICE
    config['precision'] = 'float32'
    config['experiment_path'] = str(Path(entry['experiment_path']) if entry['experiment_path'] else default_experiment_path(config))

    task = get_task(config['task'], config, DisabledLogger())
    model, siren, _ = task._build(load=False)
    load_checkpoint_pair(config, model, siren, device=task.device, suffix=entry.get('suffix', ''))
    _, renderer, grid_size = task._loss_renderer_grid()
    model.eval()
    siren.eval()
    return {
        'config': config,
        'task': task,
        'model': model,
        'siren': siren,
        'renderer': renderer,
        'grid_size': grid_size,
    }


variants = {}
for name, entry in EXPERIMENTS.items():
    try:
        variants[name] = load_variant(entry)
        print(name, 'loaded from', variants[name]['config']['experiment_path'])
    except FileNotFoundError as exc:
        print(name, 'checkpoint missing:', exc)


In [ ]:
@torch.no_grad()
def rollout(model, x, steps):
    z = None
    for _ in range(steps):
        x, z = model(x)
    return x, z


def rel_l2(a, b):
    return torch.linalg.vector_norm((a - b).reshape(a.shape[0], -1), dim=1) / (
        torch.linalg.vector_norm(b.reshape(b.shape[0], -1), dim=1) + 1e-8
    )


def affine_rotate_nchw(x, angle_deg):
    angle = torch.tensor(angle_deg * torch.pi / 180.0, device=x.device, dtype=x.dtype)
    c = torch.cos(angle)
    s = torch.sin(angle)
    theta = torch.zeros(x.shape[0], 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0] = c
    theta[:, 0, 1] = -s
    theta[:, 1, 0] = s
    theta[:, 1, 1] = c
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)


def affine_rotate_nhwc(image, angle_deg):
    x = image.permute(0, 3, 1, 2)
    x = affine_rotate_nchw(x, angle_deg)
    return x.permute(0, 2, 3, 1)


@torch.no_grad()
def render_rgba(variant, state, perception):
    model = variant['model']
    siren = variant['siren']
    renderer = variant['renderer']
    output_type = variant['config']['nca']['output_type']
    x_render = state if output_type == 's' or perception is None else perception
    image = renderer.render(x_render.permute(0, 2, 3, 1), siren, None, fs_shader='vanilla', hard_clamp=True)
    x_up = torch.nn.functional.interpolate(state.to(torch.float32), scale_factor=renderer.scale_factor, mode='bilinear')
    mask = model.get_living_mask(x_up).float().permute(0, 2, 3, 1)
    return image.to(torch.float32) * mask


In [ ]:
@torch.no_grad()
def evaluate_variant(variant, steps=STEPS, angles=ANGLES):
    model = variant['model']
    old_update_prob = getattr(model, 'update_prob', 1.0)
    model.update_prob = 1.0
    rows = []
    visuals = {}
    try:
        h, w = variant['grid_size']
        x0 = model.seed(1, h, w)
        x_base, z_base = rollout(model, x0, steps)
        image_base = render_rgba(variant, x_base, z_base)

        def add_result(kind, label, state, expected_state, image, expected_image):
            err = (image - expected_image).abs()
            rows.append({
                'kind': kind,
                'transform': label,
                'state_mae': (state - expected_state).abs().mean().item(),
                'state_rel_l2': rel_l2(state, expected_state).mean().item(),
                'render_mae': err.mean().item(),
                'render_rel_l2': rel_l2(image, expected_image).mean().item(),
            })
            visuals[label] = {
                'base': image_base.detach().cpu(),
                'transformed_seed': image.detach().cpu(),
                'transformed_output': expected_image.detach().cpu(),
                'error': err.detach().cpu(),
            }

        for k in (1, 2, 3):
            x_t0 = torch.rot90(x0, k, dims=(-2, -1))
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(variant, x_t, z_t)
            add_result(
                'exact_rotation',
                f'rot{k * 90}',
                x_t,
                torch.rot90(x_base, k, dims=(-2, -1)),
                image_t,
                torch.rot90(image_base, k, dims=(-3, -2)),
            )

        for label, dims in (('flip_x', (-1,)), ('flip_y', (-2,))):
            x_t0 = torch.flip(x0, dims=dims)
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(variant, x_t, z_t)
            image_dims = tuple(d - 1 for d in dims)
            add_result(
                'reflection',
                label,
                x_t,
                torch.flip(x_base, dims=dims),
                image_t,
                torch.flip(image_base, dims=image_dims),
            )

        for angle in angles:
            x_t0 = affine_rotate_nchw(x0, angle)
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(variant, x_t, z_t)
            add_result(
                'interpolated_rotation',
                f'rot{angle:g}',
                x_t,
                affine_rotate_nchw(x_base, angle),
                image_t,
                affine_rotate_nhwc(image_base, angle),
            )
    finally:
        model.update_prob = old_update_prob
    return rows, visuals


results = {}
visuals = {}
for name, variant in variants.items():
    rows, vis = evaluate_variant(variant)
    results[name] = rows
    visuals[name] = vis
    print()
    print(name)
    print('kind | transform | state_mae | state_rel_l2 | render_mae | render_rel_l2')
    for row in rows:
        print(
            f"{row['kind']} | {row['transform']} | {row['state_mae']:.3e} | "
            f"{row['state_rel_l2']:.3e} | {row['render_mae']:.3e} | {row['render_rel_l2']:.3e}"
        )


In [ ]:
def show_grid(variant_name, transform='rot45'):
    if variant_name not in visuals:
        print('No visuals for', variant_name)
        return
    if transform not in visuals[variant_name]:
        print('No transform', transform, 'for', variant_name)
        return

    item = visuals[variant_name][transform]
    titles = ['original render', 'rollout(transform(seed))', 'transform(rollout(seed))', 'absolute error']
    tensors = [item['base'], item['transformed_seed'], item['transformed_output'], item['error']]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, title, tensor in zip(axes, titles, tensors):
        img = tensor[0].numpy()
        if title == 'absolute error':
            ax.imshow(img.mean(axis=-1), cmap='magma')
        else:
            rgb = img[..., :3]
            alpha = img[..., 3:4]
            composite = rgb * alpha + (1.0 - alpha)
            ax.imshow(np.clip(composite, 0.0, 1.0))
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{variant_name}: {transform}')
    plt.tight_layout()
    plt.show()


for name in visuals:
    show_grid(name, 'rot45')


In [ ]:
# Try other transforms, for example:
# show_grid('coords', 'flip_x')
# show_grid('coords', 'rot90')
# show_grid('no_coords', 'rot60')
